# Explore Multi-TF Analysis Results

## Data Structure
```python
data = {
    'chrom': 'chr22',
    'variants_all': pd.DataFrame,      # All variants for annotation
    'multiTF_summary': pd.DataFrame,   # Quick reference table
    'multiTF_enhancers': {
        'k562': {
            'EH38E...': {
                'valid_clusters': ['GATA', 'ETS/1'],
                'filtered_seqlets': pd.DataFrame,  # Only seqlets from valid clusters
                'all_seqlets': pd.DataFrame,       # All seqlets (for context)
                'emvars': pd.DataFrame             # emVars with predictions
            }
        },
        'hepg2': {...},
        'sknsh': {...}
    }
}
```

In [3]:
import pickle
import pandas as pd
pd.set_option('display.max_columns', None)

In [4]:
# Load chr22 results
with open('results_per_chrom/chr1_multiTF.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"Chromosome: {data['chrom']}")
print(f"\nTop-level keys: {list(data.keys())}")

Chromosome: chr1

Top-level keys: ['chrom', 'variants_all', 'multiTF_summary', 'multiTF_enhancers']


## Quick Stats

In [5]:
print("=" * 60)
print(f"SUMMARY FOR {data['chrom']}")
print("=" * 60)
print(f"\nTotal variants in TF seqlets: {len(data['variants_all']):,}")
print(f"Variants in multi-TF enhancers: {data['variants_all']['is_multiTF'].sum():,}")
print(f"\nMulti-TF enhancers with emVars by cell type:")
for ct in ['k562', 'hepg2', 'sknsh']:
    n = len(data['multiTF_enhancers'][ct])
    print(f"  {ct}: {n:,}")
print(f"\nTotal multi-TF enhancers in summary: {len(data['multiTF_summary']):,}")

SUMMARY FOR chr1

Total variants in TF seqlets: 2,534,588
Variants in multi-TF enhancers: 501,522

Multi-TF enhancers with emVars by cell type:
  k562: 5,355
  hepg2: 5,416
  sknsh: 7,589

Total multi-TF enhancers in summary: 18,360


## List All Valid Multi-TF Enhancers

In [6]:
# Get list of all valid multi-TF enhancer IDs for a cell type
cell_type = 'k562'
valid_enhancer_ids = list(data['multiTF_enhancers'][cell_type].keys())
print(f"Valid multi-TF enhancers in {cell_type}: {len(valid_enhancer_ids)}")
print(f"\nFirst 10: {valid_enhancer_ids[:10]}")

Valid multi-TF enhancers in k562: 5355

First 10: ['EH38E2796024', 'EH38E1367043', 'EH38E2799352', 'EH38E1313912', 'EH38E2813525', 'EH38E2776575', 'EH38E2776905', 'EH38E1344689', 'EH38E2798810', 'EH38E1424230']


In [7]:
# Summary table - sorted by n_emvars
print("Multi-TF Summary (top 20 by emVar count):")
data['multiTF_summary'].head(20)

Multi-TF Summary (top 20 by emVar count):


,enhancer_id,cell_type,multiTF_clusters,n_multiTF_clusters,n_filtered_seqlets,n_all_seqlets,n_emvars,chrom,start,end
0,EH38E1428679,k562,AP1/2,1,2,13,263,chr1,227934919,227935252
1,EH38E2875836,k562,"AP1/1,Ebox/CACCTG",2,13,13,248,chr1,236097147,236097402
2,EH38E2800331,k562,Ebox/CACCTG,1,2,10,224,chr1,31549566,31549753
3,EH38E2828978,k562,Ebox/CAGCTG,1,2,9,209,chr1,108079804,108079871
4,EH38E2800196,k562,"Ebox/CACCTG,INSM1",2,4,11,206,chr1,31394782,31395045
5,EH38E1411994,k562,ZNF143,1,2,12,186,chr1,201708176,201708412
6,EH38E1344775,k562,KLF/SP/2,1,2,12,186,chr1,46527029,46527273
7,EH38E2827896,k562,"AP1/1,Ebox/CACCTG",2,4,11,177,chr1,101664870,101665089
8,EH38E2870128,k562,"Ebox/CACCTG,HIC/2",2,5,12,177,chr1,226643277,226643500
9,EH38E3980509,k562,AP1/1,1,3,10,174,chr1,156398556,156398644


## Explore a Specific Enhancer

In [8]:
# Pick an enhancer to explore
cell_type = 'k562'

if len(data['multiTF_enhancers'][cell_type]) > 0:
    # Get first enhancer from summary (highest emVar count)
    summary_ct = data['multiTF_summary'][data['multiTF_summary']['cell_type'] == cell_type]
    if len(summary_ct) > 0:
        enhancer_id = summary_ct.iloc[0]['enhancer_id']
    else:
        enhancer_id = list(data['multiTF_enhancers'][cell_type].keys())[0]
    
    print(f"Exploring enhancer: {enhancer_id} ({cell_type})")
else:
    print(f"No multi-TF enhancers found for {cell_type}")
    enhancer_id = None

Exploring enhancer: EH38E1428679 (k562)


In [9]:
# Access enhancer data directly
if enhancer_id:
    enh_data = data['multiTF_enhancers'][cell_type][enhancer_id]
    
    print(f"Valid clusters: {enh_data['valid_clusters']}")
    print(f"\nFiltered seqlets (only from valid clusters): {len(enh_data['filtered_seqlets'])} intervals")
    print(f"All seqlets (for context): {len(enh_data['all_seqlets'])} intervals")
    print(f"EmVars: {len(enh_data['emvars'])} rows")

Valid clusters: ['AP1/2']

Filtered seqlets (only from valid clusters): 2 intervals
All seqlets (for context): 13 intervals
EmVars: 263 rows


In [10]:
# View FILTERED seqlets - these are the ones that made it a valid multi-TF enhancer
if enhancer_id:
    print(f"FILTERED SEQLETS for {enhancer_id}:")
    print(f"(Only seqlets from valid clusters: {enh_data['valid_clusters']})")
    print()
    display(enh_data['filtered_seqlets'][['chrom', 'start', 'end', 'representative_tf', 'vierstra_cluster', 'rep_tf_contrib', 'activity_class']])

FILTERED SEQLETS for EH38E1428679:
(Only seqlets from valid clusters: ['AP1/2'])



,chrom,start,end,representative_tf,vierstra_cluster,rep_tf_contrib,activity_class
112948,chr1,227934950,227934960,NF2L2,AP1/2,8.937465,Activator
112955,chr1,227935107,227935117,NF2L2,AP1/2,9.482644,Activator


In [11]:
# View ALL seqlets for context
if enhancer_id:
    print(f"ALL SEQLETS for {enhancer_id} (for context):")
    print()
    display(enh_data['all_seqlets'][['chrom', 'start', 'end', 'representative_tf', 'vierstra_cluster', 'rep_tf_contrib', 'activity_class']])

ALL SEQLETS for EH38E1428679 (for context):



,chrom,start,end,representative_tf,vierstra_cluster,rep_tf_contrib,activity_class
112946,chr1,227934919,227934927,FOSL2,AP1/1,6.871444,Activator
112947,chr1,227934932,227934937,SNAI1,Ebox/CACCTG,-1.018541,Repressor
112948,chr1,227934950,227934960,NF2L2,AP1/2,8.937465,Activator
112949,chr1,227934982,227934992,FOSL1,AP1/1,9.104137,Activator
112950,chr1,227935014,227935024,FOSL1,AP1/1,8.769096,Activator
112951,chr1,227935014,227935034,FOSL1,AP1/1,8.769096,Activator
112952,chr1,227935015,227935034,FOSL1,AP1/1,8.769096,Activator
112953,chr1,227935044,227935054,FOSL1,AP1/1,10.216508,Activator
112954,chr1,227935076,227935087,FOSL1,AP1/1,9.837755,Activator
112955,chr1,227935107,227935117,NF2L2,AP1/2,9.482644,Activator


In [12]:
# View emVars with predictions
if enhancer_id:
    print(f"EMVARS for {enhancer_id}:")
    print()
    emvar_cols = ['variant_id', 'skew_pred', 'chrom', 'start', 'end']
    display(enh_data['emvars'][emvar_cols].head(20))

EMVARS for EH38E1428679:



,variant_id,skew_pred,chrom,start,end
905179,chr1:227934920:T:A,-1.052097,chr1,227934919,227934927
905180,chr1:227934920:T:C,-1.019040,chr1,227934919,227934927
905181,chr1:227934920:T:G,-0.966040,chr1,227934919,227934927
905182,chr1:227934921:G:C,-1.120445,chr1,227934919,227934927
905183,chr1:227934921:G:A,-1.132720,chr1,227934919,227934927
905184,chr1:227934922:A:T,-0.900724,chr1,227934919,227934927
905185,chr1:227934922:A:G,-1.178971,chr1,227934919,227934927
905186,chr1:227934923:G:A,-2.075320,chr1,227934919,227934927
905187,chr1:227934923:G:T,-1.044013,chr1,227934919,227934927
905188,chr1:227934924:T:C,-1.059104,chr1,227934919,227934927


## Filter by TF Family

In [13]:
# Find enhancers with a specific TF family
tf_family = 'GATA'  # Change this

filtered = data['multiTF_summary'][
    data['multiTF_summary']['multiTF_clusters'].str.contains(tf_family, na=False)
]
print(f"Enhancers with multiple non-overlapping {tf_family} instances: {len(filtered)}")
filtered.head(10)

Enhancers with multiple non-overlapping GATA instances: 351


,enhancer_id,cell_type,multiTF_clusters,n_multiTF_clusters,n_filtered_seqlets,n_all_seqlets,n_emvars,chrom,start,end
13,EH38E2842657,k562,GATA,1,2,10,163,chr1,160046586,160046875
25,EH38E2832658,k562,"AP1/1,GATA",2,5,6,144,chr1,115179536,115179742
27,EH38E2812713,k562,"Ebox/CACCTG,GATA",2,4,10,142,chr1,54409331,54409645
33,EH38E2827684,k562,GATA,1,2,9,139,chr1,101071574,101071849
38,EH38E1326159,k562,GATA,1,2,7,138,chr1,21491664,21491740
40,EH38E2793049,k562,"ETS/1,GATA,INSM1",3,7,10,137,chr1,21632196,21632480
59,EH38E2848532,k562,GATA,1,2,6,126,chr1,173355318,173355469
77,EH38E2799740,k562,"GATA,GLI",2,4,12,120,chr1,30769026,30769277
81,EH38E1375766,k562,"AP1/1,GATA",2,4,8,118,chr1,110477254,110477421
107,EH38E2809615,k562,GATA,1,2,12,112,chr1,47181259,47181479


## Variants DataFrame (for annotation)

In [14]:
# Variants ready for annotation
print("VARIANTS_ALL - for annotation pipelines:")
print(f"Columns: {data['variants_all'].columns.tolist()}")
print(f"\nShape: {data['variants_all'].shape}")
data['variants_all'].head(10)

VARIANTS_ALL - for annotation pipelines:
Columns: ['variant_id', 'chrom', 'pos', 'ref', 'alt', 'cell_type', 'skew_pred', 'enhancer_ids', 'is_multiTF']

Shape: (2534588, 9)


,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF
0,chr1:10001330:G:A,chr1,10001330,G,A,k562,1.087750,EH38E1317660,False
1,chr1:100028994:T:A,chr1,100028994,T,A,k562,-0.507599,EH38E3974467,False
2,chr1:100028994:T:C,chr1,100028994,T,C,k562,-0.621325,EH38E3974467,False
3,chr1:100028994:T:G,chr1,100028994,T,G,k562,-0.632198,EH38E3974467,False
4,chr1:100028995:A:C,chr1,100028995,A,C,k562,-0.782038,EH38E3974467,False
5,chr1:100028995:A:G,chr1,100028995,A,G,k562,-0.661248,EH38E3974467,False
6,chr1:100028995:A:T,chr1,100028995,A,T,k562,-0.736603,EH38E3974467,False
7,chr1:100028996:T:A,chr1,100028996,T,A,k562,-0.767162,EH38E3974467,False
8,chr1:100028996:T:C,chr1,100028996,T,C,k562,-0.732322,EH38E3974467,False
9,chr1:100028996:T:G,chr1,100028996,T,G,k562,-0.753687,EH38E3974467,False


In [13]:
# Breakdown by cell type
print("Variants by cell type:")
print(data['variants_all'].groupby('cell_type').size())
print("\nVariants in multi-TF enhancers:")
print(data['variants_all'][data['variants_all']['is_multiTF']].groupby('cell_type').size())

Variants by cell type:
cell_type
hepg2    168471
k562     215286
sknsh    198191
dtype: int64

Variants in multi-TF enhancers:
cell_type
hepg2    25701
k562     42123
sknsh    43586
dtype: int64


In [15]:
data['multiTF_enhancers']['k562']['EH38E1428679']['filtered_seqlets']

,chrom,start,end,name,score,strand,thickStart,thickEnd,full_hit_string,hocomoco_tf,representative_tf,enhancer_id,rep_tf_contrib,vierstra_cluster,activity_class
112948,chr1,227934950,227934960,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_8.9374...,1,NF2L2,Activator,EH38E1428679,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_8.9374...,NF2L2_HUMAN.H11MO.0.A,NF2L2,EH38E1428679,8.937465,AP1/2,Activator
112955,chr1,227935107,227935117,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_9.4826...,1,NF2L2,Activator,EH38E1428679,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_9.4826...,NF2L2_HUMAN.H11MO.0.A,NF2L2,EH38E1428679,9.482644,AP1/2,Activator


In [16]:
data['multiTF_enhancers']['k562']['EH38E1428679']['all_seqlets']

,chrom,start,end,name,score,strand,thickStart,thickEnd,full_hit_string,hocomoco_tf,representative_tf,enhancer_id,rep_tf_contrib,vierstra_cluster,activity_class
112946,chr1,227934919,227934927,FOSL2_HUMAN.H11MO.0.A_EH38E1428679_K562_6.8714...,1,FOSL2,Activator,EH38E1428679,FOSL2_HUMAN.H11MO.0.A_EH38E1428679_K562_6.8714...,FOSL2_HUMAN.H11MO.0.A,FOSL2,EH38E1428679,6.871444,AP1/1,Activator
112947,chr1,227934932,227934937,SNAI1_HUMAN.H11MO.0.C_EH38E1428679_K562_-1.018...,1,SNAI1,Repressor,EH38E1428679,SNAI1_HUMAN.H11MO.0.C_EH38E1428679_K562_-1.018...,SNAI1_HUMAN.H11MO.0.C,SNAI1,EH38E1428679,-1.018541,Ebox/CACCTG,Repressor
112948,chr1,227934950,227934960,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_8.9374...,1,NF2L2,Activator,EH38E1428679,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_8.9374...,NF2L2_HUMAN.H11MO.0.A,NF2L2,EH38E1428679,8.937465,AP1/2,Activator
112949,chr1,227934982,227934992,MAFG_HUMAN.H11MO.0.A_EH38E1428679_K562_3.14034...,2,FOSL1,Activator,EH38E1428679,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_9.1041...,FOSL1_HUMAN.H11MO.0.A,FOSL1,EH38E1428679,9.104137,AP1/1,Activator
112950,chr1,227935014,227935024,MAFG_HUMAN.H11MO.0.A_EH38E1428679_K562_3.02141...,2,FOSL1,Activator,EH38E1428679,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_8.7690...,FOSL1_HUMAN.H11MO.0.A,FOSL1,EH38E1428679,8.769096,AP1/1,Activator
112951,chr1,227935014,227935034,MAFG_HUMAN.H11MO.0.A_EH38E1428679_K562_3.02141...,3,FOSL1,Activator,EH38E1428679,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_8.7690...,FOSL1_HUMAN.H11MO.0.A,FOSL1,EH38E1428679,8.769096,AP1/1,Activator
112952,chr1,227935015,227935034,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_8.7690...,2,FOSL1,Activator,EH38E1428679,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_8.7690...,FOSL1_HUMAN.H11MO.0.A,FOSL1,EH38E1428679,8.769096,AP1/1,Activator
112953,chr1,227935044,227935054,MAFG_HUMAN.H11MO.0.A_EH38E1428679_K562_3.70373...,2,FOSL1,Activator,EH38E1428679,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_10.216...,FOSL1_HUMAN.H11MO.0.A,FOSL1,EH38E1428679,10.216508,AP1/1,Activator
112954,chr1,227935076,227935087,MAFG_HUMAN.H11MO.0.A_EH38E1428679_K562_3.34906...,2,FOSL1,Activator,EH38E1428679,FOSL1_HUMAN.H11MO.0.A_EH38E1428679_K562_9.8377...,FOSL1_HUMAN.H11MO.0.A,FOSL1,EH38E1428679,9.837755,AP1/1,Activator
112955,chr1,227935107,227935117,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_9.4826...,1,NF2L2,Activator,EH38E1428679,NF2L2_HUMAN.H11MO.0.A_EH38E1428679_K562_9.4826...,NF2L2_HUMAN.H11MO.0.A,NF2L2,EH38E1428679,9.482644,AP1/2,Activator


In [33]:
data['multiTF_enhancers']['k562']['EH38E2169799']['emvars']

,chrom,start,end,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts,variant_id,var_chrom,var_pos,var_ref,var_alt,pos,ref,alt,skew_pred
184705,chr22,45703126,45703137,ZN586_HUMAN.H11MO.0.C_EH38E2169799_K562_-1.164...,3,ZN121,Repressor,EH38E2169799,chr22,45703128,45703129,chr22:45703129:T:G_EH38E2169799,chr22:45703129:T:G,chr22,45703129,T,G,45703129,T,G,0.615922
184706,chr22,45703126,45703137,ZN586_HUMAN.H11MO.0.C_EH38E2169799_K562_-1.164...,3,ZN121,Repressor,EH38E2169799,chr22,45703128,45703129,chr22:45703129:T:C_EH38E2169799,chr22:45703129:T:C,chr22,45703129,T,C,45703129,T,C,0.953511
184707,chr22,45703126,45703137,ZN586_HUMAN.H11MO.0.C_EH38E2169799_K562_-1.164...,3,ZN121,Repressor,EH38E2169799,chr22,45703130,45703131,chr22:45703131:C:A_EH38E2169799,chr22:45703131:C:A,chr22,45703131,C,A,45703131,C,A,1.034776
184708,chr22,45703126,45703137,ZN586_HUMAN.H11MO.0.C_EH38E2169799_K562_-1.164...,3,ZN121,Repressor,EH38E2169799,chr22,45703131,45703132,chr22:45703132:C:G_EH38E2169799,chr22:45703132:C:G,chr22,45703132,C,G,45703132,C,G,1.155662
184709,chr22,45703126,45703137,ZN586_HUMAN.H11MO.0.C_EH38E2169799_K562_-1.164...,3,ZN121,Repressor,EH38E2169799,chr22,45703131,45703132,chr22:45703132:C:T_EH38E2169799,chr22:45703132:C:T,chr22,45703132,C,T,45703132,C,T,0.690739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184768,chr22,45703232,45703237,TAL1_HUMAN.H11MO.0.A_EH38E2169799_K562_3.11009...,1,TAL1,Activator,EH38E2169799,chr22,45703234,45703235,chr22:45703235:T:A_EH38E2169799,chr22:45703235:T:A,chr22,45703235,T,A,45703235,T,A,-0.733509
184769,chr22,45703232,45703237,TAL1_HUMAN.H11MO.0.A_EH38E2169799_K562_3.11009...,1,TAL1,Activator,EH38E2169799,chr22,45703235,45703236,chr22:45703236:A:G_EH38E2169799,chr22:45703236:A:G,chr22,45703236,A,G,45703236,A,G,-0.825359
184770,chr22,45703232,45703237,TAL1_HUMAN.H11MO.0.A_EH38E2169799_K562_3.11009...,1,TAL1,Activator,EH38E2169799,chr22,45703235,45703236,chr22:45703236:A:C_EH38E2169799,chr22:45703236:A:C,chr22,45703236,A,C,45703236,A,C,-0.743100
184771,chr22,45703232,45703237,TAL1_HUMAN.H11MO.0.A_EH38E2169799_K562_3.11009...,1,TAL1,Activator,EH38E2169799,chr22,45703235,45703236,chr22:45703236:A:T_EH38E2169799,chr22:45703236:A:T,chr22,45703236,A,T,45703236,A,T,-0.748455
